# Chapter 38: Support Vector Machines, Nearest Neighbours, and Kernel Methods

Synthetic NRG shipment data show how scaling changes distance and margin models.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.distance_margin import fit_scaler,apply_scaler,fit_with_seed,binary_report,support_summary,nearest_indices
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(38);n=420
distance=rng.uniform(100,2500,n);wait=rng.uniform(0,16,n)
late=((distance/900-1.4)**2+(wait/5-1.5)**2+rng.normal(0,.55,n)>2.0).astype(int)
x=np.column_stack([distance,wait]);cut=320
mean,scale=fit_scaler(x[:cut]);z_train=apply_scaler(x[:cut],mean,scale);z_test=apply_scaler(x[cut:],mean,scale)
print(f'Train rows: {cut}; test rows: {n-cut}; test prevalence: {late[cut:].mean():.1%}')


Train rows: 320; test rows: 100; test prevalence: 36.0%


In [ ]:
models={'KNN':KNeighborsClassifier(n_neighbors=17,weights='distance'),'linear SVM':SVC(kernel='linear',probability=True,C=1),'RBF SVM':SVC(kernel='rbf',probability=True,C=3,gamma=.7)}
fitted={}
for name,model in models.items():
 fitted[name]=fit_with_seed(model,z_train,late[:cut]);report=binary_report(fitted[name],z_test,late[cut:]);print(name,f"accuracy={report['accuracy']:.3f}",f"AUC={report['roc_auc']:.3f}")


KNN accuracy=0.880 AUC=0.929
linear SVM accuracy=0.640 AUC=0.571
RBF SVM accuracy=0.890 AUC=0.932


In [ ]:
print('Linear support vectors:',support_summary(fitted['linear SVM']))
print('RBF support vectors:',support_summary(fitted['RBF SVM']))
idx=nearest_indices(z_test[0],z_train,5);print('First test row neighbour labels:',late[idx].tolist())


Linear support vectors: {'total': 197, 'by_class': [101, 96]}
RBF support vectors: {'total': 100, 'by_class': [50, 50]}
First test row neighbour labels: [0, 0, 0, 0, 1]


In [ ]:
gx,gy=np.meshgrid(np.linspace(z_train[:,0].min(),z_train[:,0].max(),150),np.linspace(z_train[:,1].min(),z_train[:,1].max(),150));grid=np.column_stack([gx.ravel(),gy.ravel()])
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,name in zip(axes,['KNN','RBF SVM']):
 score=fitted[name].predict_proba(grid)[:,1].reshape(gx.shape);ax.contourf(gx,gy,score,levels=np.linspace(0,1,11),cmap='RdYlBu_r',alpha=.65);ax.scatter(z_test[:,0],z_test[:,1],c=late[cut:],cmap='bwr',s=14,edgecolor='white',linewidth=.2);ax.set(title=name,xlabel='Scaled distance',ylabel='Scaled wait')
fig.tight_layout();plt.show()


## Interpretation

Scaling is fitted on training rows. KNN forms a local boundary, a linear SVM uses one straight margin, and the RBF kernel permits a curved boundary. Final selection also requires calibration, cost, capacity, latency, and stability.


In [ ]:
# Practice: vary k, C, and gamma within cross-validation.
